# 25 — OpenRouter ReAct Agent
> A question goes in; the agent decides which tool to call, runs it, reads the result, and keeps going until it has a final answer — showing every step along the way.

*Run all cells. Swap the input in the final code cell with your own data.*

In [ ]:
%pip install -q openai python-dotenv
import os
os.environ['OPENROUTER_API_KEY'] = 'your-openrouter-key'  # replace

In [ ]:
import json
import os
from typing import Any
from pydantic import BaseModel, Field
from openai import OpenAI


# --- Data models ---

class AgentStep(BaseModel):
    step: int = Field(description="Step index in the loop (1-based).")
    action: str = Field(description="Tool name called, or 'final' for the last answer.")
    input: str = Field(description="Arguments passed to the tool or the final answer text.")
    observation: str = Field(description="Tool result, or empty string for the final answer step.")


# --- Tool definitions sent to the model ---

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "add",
            "description": "Add two numbers together.",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "number", "description": "First number."},
                    "b": {"type": "number", "description": "Second number."},
                },
                "required": ["a", "b"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "multiply",
            "description": "Multiply two numbers together.",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "number", "description": "First number."},
                    "b": {"type": "number", "description": "Second number."},
                },
                "required": ["a", "b"],
            },
        },
    },
]

# --- Local tool implementations ---

DISPATCH: dict[str, Any] = {
    "add": lambda a, b: a + b,
    "multiply": lambda a, b: a * b,
}


# --- Agent loop ---

def run(question: str, model: str = "openai/gpt-4o-mini") -> tuple[str, list[AgentStep]]:
    """
    Run the agent loop for a math question.

    Args:
        question: Natural-language math question.
        model: OpenRouter model identifier string.

    Returns:
        Tuple of (final_answer, trace) where trace lists every step taken.
    """
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.environ["OPENROUTER_API_KEY"],
    )
    messages: list[dict] = [
        {
            "role": "system",
            "content": (
                "You are a math assistant. Solve problems using only the provided tools. "
                "Do not compute answers yourself — always call a tool."
            ),
        },
        {"role": "user", "content": question},
    ]

    trace: list[AgentStep] = []
    step = 0

    while True:
        step += 1
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=TOOLS,
            tool_choice="auto",
        )
        message = response.choices[0].message

        if not message.tool_calls:
            final = message.content or ""
            trace.append(AgentStep(step=step, action="final", input=final, observation=""))
            return final, trace

        messages.append(message.model_dump(exclude_unset=True))

        for call in message.tool_calls:
            name = call.function.name
            args = json.loads(call.function.arguments)
            result = DISPATCH[name](**args)
            observation = str(result)

            trace.append(
                AgentStep(
                    step=step,
                    action=name,
                    input=call.function.arguments,
                    observation=observation,
                )
            )
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": call.id,
                    "content": observation,
                }
            )

## The scenario

A project manager needs to calculate the total budget for a team sprint: 5 developers each billing 8 hours a day for 9 days at $95/hr, plus a flat $250 tooling fee. The agent works through each calculation in sequence — multiplying, then adding — and shows exactly which operation it ran at each stage before landing on the final number.

In [ ]:
question = (
    "I have 5 developers each working 8 hours a day for 9 days at $95 per hour. "
    "There is also a flat tooling fee of $250. What is the total project cost?"
)

answer, trace = run(question)

print(f"Question: {question}")
print("-" * 60)
print("Reasoning trace:")
for s in trace:
    if s.action == "final":
        print(f"  [step {s.step}] ANSWER  ->  {s.input}")
    else:
        print(f"  [step {s.step}] {s.action}({s.input})  =  {s.observation}")
print("-" * 60)
print(f"Final answer: {answer}")

## Use your own data

Replace the `question` string above with:
- Your own multi-step arithmetic problem stated in plain English
- Any quantity, rate, or cost calculation you would normally reach for a spreadsheet to solve

The agent will return the final number along with every intermediate calculation it ran to get there.